# Exploratory Data Analysis

Looks at the raw Agriculture Vision dataset before building anything: folder structure, label
encoding, and class balance. Confirms drydown is a binary mask, boundaries and masks are separate
valid area masks and not model input, and there is enough drydown signal in both train and val to
support a balanced, field disjoint sample.

# 1. Setup

## 1.1 Mount Drive

Mounts Drive so the notebook can save any exported files.

In [2]:
# Mounts Google Drive at /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1.2 Go to the Project Folder

In [4]:
# Working directory for this session
%cd /content/drive/MyDrive/Rizoma

/content/drive/MyDrive/Rizoma


In [6]:
# Confirm the project folder's contents before proceeding
!ls

'Agriculture-Vision dataset'	   'Rizoma .docx'
 DeepLabV3_Drydown		   'Ultralytics YOLO26'
'Exploratory Data Analysis (EDA)'


## 1.3 Install the AWS CLI

Needed to pull the dataset from S3.

In [7]:
# awscli lets us pull the dataset from S3 without a full SDK
!pip install -q awscli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 28.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [8]:
# Confirm the CLI installed correctly
!aws --version

aws-cli/1.46.1 Python/3.13.15 Linux/6.6.122+ botocore/1.43.62


## 1.4 Make a Folder for the Archive

## 1.5 Download the Dataset

Pulls the official release from its public S3 bucket, no AWS account needed. Licence and citation
are in Methodology.md.

In [10]:
# Anonymous S3 download (no AWS credentials needed)
!aws s3 cp s3://intelinair-data-releases/agriculture-vision/cvpr_challenge_2021/supervised/Agriculture-Vision-2021.tar.gz "/content/Agriculture-Vision-2021.tar.gz" --no-sign-request

download: s3://intelinair-data-releases/agriculture-vision/cvpr_challenge_2021/supervised/Agriculture-Vision-2021.tar.gz to ../../../Agriculture-Vision-2021.tar.gz


In [11]:
# Extract to local Colab disk (fast, ephemeral) rather than Drive
!tar -xzf "/content/Agriculture-Vision-2021.tar.gz" -C /content

In [12]:
# Simple marker that extraction finished
print("DONE")

DONE


# 2. Dataset Structure

## 2.1 Top Level Folders

Confirms images and labels are structured as expected.

In [13]:
# Expect: nir, rgb
!ls /content/Agriculture-Vision-2021/train/images

nir  rgb


In [14]:
# Expect the 9 documented anomaly classes, including drydown
!ls /content/Agriculture-Vision-2021/train/labels

double_plant  endrow		   planter_skip  water	   weed_cluster
drydown       nutrient_deficiency  storm_damage  waterway


## 2.2 Check One Drydown Mask

This one is all zero, no drydown in this tile, which is expected since drydown is a minority
class.

In [15]:
# This file's mask is empty (no drydown here) — expected for most tiles
from PIL import Image
import numpy as np
m = np.array(Image.open("/content/Agriculture-Vision-2021/train/labels/drydown/11IE4DKTR_11556-9586-12068-10098.png"))
print(m.shape, m.dtype, np.unique(m))

(512, 512) uint8 [0]


In [16]:
# Sample the first 5 files alphabetically to confirm the all-zero pattern
# isn't a one-off — this is normal, not a bug
import os
from PIL import Image
import numpy as np

folder = "/content/Agriculture-Vision-2021/train/labels/drydown"
for f in sorted(os.listdir(folder))[:5]:
    m = np.array(Image.open(os.path.join(folder, f)))
    print(f, m.dtype, np.unique(m))

11IE4DKTR_11556-9586-12068-10098.png uint8 [0]
11IE4DKTR_6121-684-6633-1196.png uint8 [0]
11TAQJ6ET_1124-8510-1636-9022.png uint8 [0]
11TAQJ6ET_1148-8089-1660-8601.png uint8 [0]
11TAQJ6ET_1288-961-1800-1473.png uint8 [0]


## 2.3 Find a Positive Example

Confirms the encoding: 0 is background, 255 is drydown.

In [17]:
# Scan until the first file with any nonzero pixels is found
import os
from PIL import Image
import numpy as np

folder = "/content/Agriculture-Vision-2021/train/labels/drydown"
files = sorted(os.listdir(folder))
for f in files:
    m = np.array(Image.open(os.path.join(folder, f)))
    if m.max() > 0:
        print(f, m.dtype, np.unique(m))
        break

167KTRRUZ_1499-5040-2011-5552.png uint8 [  0 255]


# 3. Size on Disk

## 3.1 Total Footprint

How much space the extracted dataset takes locally.

In [18]:
# Extracted dataset footprint on local disk
!du -sh /content/Agriculture-Vision-2021

24G	/content/Agriculture-Vision-2021


# 4. File Verification

## 4.1 One File Per Image, Everywhere

Checks that every folder, images, boundaries, masks, and each label class, has the same file
count, and that files pair up by name alone.

In [19]:
# All folders should report the same count
!for d in images/rgb images/nir boundaries masks labels/drydown labels/nutrient_deficiency labels/weed_cluster; do echo -n "$d: "; ls /content/Agriculture-Vision-2021/train/$d | wc -l; done

images/rgb: 56944
images/nir: 56944
boundaries: 56944
masks: 56944
labels/drydown: 56944
labels/nutrient_deficiency: 56944
labels/weed_cluster: 56944


In [20]:
# 4.2 Confirm filenames match across folders: same stem, different extensions
# (.jpg for images, .png for masks/labels) — files can be paired by name alone
!for d in images/rgb images/nir boundaries masks labels/drydown; do f=$(ls /content/Agriculture-Vision-2021/train/$d | head -1); echo "$d -> $f"; done

images/rgb -> 11IE4DKTR_11556-9586-12068-10098.jpg
images/nir -> 11IE4DKTR_11556-9586-12068-10098.jpg
boundaries -> 11IE4DKTR_11556-9586-12068-10098.png
masks -> 11IE4DKTR_11556-9586-12068-10098.png
labels/drydown -> 11IE4DKTR_11556-9586-12068-10098.png


In [21]:
# 4.3 Distinguish `boundaries` (field polygon: has both 0 and 255 here) from
# `masks` (valid sensor data: all-255 here, fully valid) — two different concepts
import numpy as np
from PIL import Image
b = np.array(Image.open("/content/Agriculture-Vision-2021/train/boundaries/11IE4DKTR_11556-9586-12068-10098.png"))
m = np.array(Image.open("/content/Agriculture-Vision-2021/train/masks/11IE4DKTR_11556-9586-12068-10098.png"))
print("boundaries:", b.shape, b.dtype, np.unique(b))
print("masks:", m.shape, m.dtype, np.unique(m))

boundaries: (512, 512) uint8 [  0 255]
masks: (512, 512) uint8 [255]


# 5. Drydown Coverage

Checks how much of each image is drydown, in train and val, and counts unique fields in val.

In [22]:
# 5.1 Train split: coverage = drydown_pixels / total_pixels, per image
import os
import numpy as np
from PIL import Image

folder = "/content/Agriculture-Vision-2021/train/labels/drydown"
files = os.listdir(folder)
coverage = []
for f in files:
    m = np.array(Image.open(os.path.join(folder, f)))
    pct = (m > 0).mean()
    if pct > 0:
        coverage.append(pct)

print("images with any drydown:", len(coverage), "/", len(files))
print("images with >=5% drydown:", sum(1 for c in coverage if c >= 0.05))
print("images with >=20% drydown:", sum(1 for c in coverage if c >= 0.20))

images with any drydown: 16806 / 56944
images with >=5% drydown: 15773
images with >=20% drydown: 12582


In [23]:
# 5.2 Val split: same coverage check, to see how many usable positive/negative
# examples exist in the pool used for validation + test
import os
import numpy as np
from PIL import Image

folder = "/content/Agriculture-Vision-2021/val/labels/drydown"
files = os.listdir(folder)
coverage = []
for f in files:
    m = np.array(Image.open(os.path.join(folder, f)))
    pct = (m > 0).mean()
    if pct > 0:
        coverage.append(pct)

print("val total:", len(files))
print("val images with any drydown:", len(coverage))
print("val images with >=20% drydown:", sum(1 for c in coverage if c >= 0.20))

val total: 18334
val images with any drydown: 5800
val images with >=20% drydown: 4493


In [24]:
# 5.3 Val's field count matters because the val/test split (Methodology.md)
# divides these FIELDS 50/50 (not individual images), to stay field-disjoint
import os
files = os.listdir("/content/Agriculture-Vision-2021/val/labels/drydown")
field_ids = set(f.split("_")[0] for f in files)
print("unique fields in val:", len(field_ids))

unique fields in val: 656


## 5.4 Positive and Negative Counts Per Half

Checks how many positive and negative candidates fall into each half of a 50/50 field split,
before assuming that split is safe to use. Drydown positive images are not spread evenly across
fields, so the two halves might not end up balanced. They did not, which is why the real training
notebooks use a different, balanced split instead.

In [26]:
import os, random
import numpy as np
from PIL import Image

random.seed(42)  # matches the training notebook's CONFIG["seed"] for a representative check

folder = "/content/Agriculture-Vision-2021/val/labels/drydown"
files = os.listdir(folder)

coverage = {}
for f in files:
    m = np.array(Image.open(os.path.join(folder, f)))
    coverage[f] = (m > 0).mean()

field_ids = sorted(set(f.split("_")[0] for f in files))
random.shuffle(field_ids)
half = len(field_ids) // 2
val_fields = set(field_ids[:half])
test_fields = set(field_ids[half:])

def count_candidates(fields):
    pos = sum(1 for f, c in coverage.items() if c >= 0.20 and f.split("_")[0] in fields)
    neg = sum(1 for f, c in coverage.items() if c == 0 and f.split("_")[0] in fields)
    return pos, neg

val_pos, val_neg = count_candidates(val_fields)
test_pos, test_neg = count_candidates(test_fields)

print(f"validation half: {len(val_fields)} fields -> {val_pos} positive, {val_neg} negative candidates")
print(f"test half:       {len(test_fields)} fields -> {test_pos} positive, {test_neg} negative candidates")

validation half: 328 fields -> 2768 positive, 7476 negative candidates
test half:       328 fields -> 1725 positive, 5058 negative candidates


# 6. Visual Check

## 6.1 RGB, False Color, and Mask Side by Side

Shows one mild, one medium, and one extreme drydown example each run: true RGB, a false color
NIR view, and the mask overlay.

In [25]:
# Re-run this cell for a different random mild/medium/extreme trio each time
import os, random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ROOT = "/content/Agriculture-Vision-2021/train"
mask_dir = f"{ROOT}/labels/drydown"
rgb_dir = f"{ROOT}/images/rgb"
nir_dir = f"{ROOT}/images/nir"

files = os.listdir(mask_dir)
random.shuffle(files)

mild, medium, extreme = None, None, None
for f in files:
    m = np.array(Image.open(os.path.join(mask_dir, f)))
    pct = (m > 0).mean()
    if 0 < pct <= 0.05 and mild is None:
        mild = (f, pct)
    elif 0.05 < pct < 0.20 and medium is None:
        medium = (f, pct)
    elif pct >= 0.20 and extreme is None:
        extreme = (f, pct)
    if mild and medium and extreme:
        break

rows = [mild, medium, extreme]
row_labels = ["Mild (<=5%)", "Medium (5-20%)", "Extreme (>=20%)"]

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
for row, (fname, pct) in enumerate(rows):
    rgb = np.array(Image.open(os.path.join(rgb_dir, fname.replace(".png", ".jpg"))))
    nir = np.array(Image.open(os.path.join(nir_dir, fname.replace(".png", ".jpg"))).convert("L"))
    mask = np.array(Image.open(os.path.join(mask_dir, fname)))

    # Column 1: true RGB
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"{row_labels[row]} — {pct*100:.1f}% drydown\nRGB")
    axes[row, 0].axis("off")

    # Column 2: false-color NIR composite (NIR, R, G) — standard way to view a 4-channel image as one picture
    false_color = np.stack([nir, rgb[:, :, 0], rgb[:, :, 1]], axis=-1)
    axes[row, 1].imshow(false_color)
    axes[row, 1].set_title("False-color (NIR-R-G)\n= 4-channel view")
    axes[row, 1].axis("off")

    # Column 3: segmentation mask overlay
    overlay = rgb.copy()
    overlay[mask > 0] = [255, 0, 0]
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title("Segmentation\n(drydown highlighted)")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.